# Airbnb Feature Pipeline: Engenharia de Atributos com PySpark

### Desenvolvimento de Pipeline e Preparação de Dados em Larga Escala com Spark

In [1]:
# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [2]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "André Luiz Colombo"

Author: André Luiz Colombo



In [3]:
#Inicialização do ambiente Spark configurado para rodar localmente utilizando todos os cores disponíveis (local[*])
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("FeatureEngineeringAirbnb") \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-22T01:37:05,229 [Thread-3] WARN  org.apache.hadoop.util.NativeCodeLoader [] - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Leitura otimizada do CSV tratando quebras de linha em colunas de texto (multiLine) e caracteres de escape das aspas
df = (spark.read.csv("/opt/spark/resources/NYC/base/AB_NYC_2019.csv", 
    header=True, 
    inferSchema=True, 
    multiLine=True, 
    escape='"'))

# Exibição da estrutura de dados e validação dos tipos primitivos atribuídos pelo Spark
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: date (nullable = true)
 |-- reviews_per_month: double (nullable = true)
 |-- calculated_host_listings_count: integer (nullable = true)
 |-- availability_365: integer (nullable = true)



In [5]:
# Validação visual dos dados carregados sem truncamento das strings para inspecionar a integridade das colunas de texto
df.show(truncate=False)

+----+------------------------------------------------+-------+----------------+-------------------+------------------+--------+---------+---------------+-----+--------------+-----------------+-----------+-----------------+------------------------------+----------------+
|id  |name                                            |host_id|host_name       |neighbourhood_group|neighbourhood     |latitude|longitude|room_type      |price|minimum_nights|number_of_reviews|last_review|reviews_per_month|calculated_host_listings_count|availability_365|
+----+------------------------------------------------+-------+----------------+-------------------+------------------+--------+---------+---------------+-----+--------------+-----------------+-----------+-----------------+------------------------------+----------------+
|2539|Clean & quiet apt home by the park              |2787   |John            |Brooklyn           |Kensington        |40.64749|-73.97237|Private room   |149  |1             |9        

In [6]:
# Engenharia de Atributos sobre Texto: Extração de tamanho do título e flags booleanas via Regex para identificar padrões de negócio
df_texto = df.withColumn(
    "name_length", 
    F.length(F.col("name"))
).withColumn(
    "is_luxury", 
    F.col("name").rlike("(?i)luxury|penthouse|loft") # (?i) serve para ignorar maiúsculas/minúsculas
).withColumn(
    "has_wifi", 
    F.col("name").rlike("(?i)wifi|wi-fi")
)

In [7]:
# Inspeção das novas variáveis criadas para validar os padrões de texto extraídos e os comprimentos calculadosd
df_texto.select("name", "name_length", "is_luxury", "has_wifi").show(truncate=False)

+------------------------------------------------+-----------+---------+--------+
|name                                            |name_length|is_luxury|has_wifi|
+------------------------------------------------+-----------+---------+--------+
|Clean & quiet apt home by the park              |34         |false    |false   |
|Skylit Midtown Castle                           |21         |false    |false   |
|THE VILLAGE OF HARLEM....NEW YORK !             |35         |false    |false   |
|Cozy Entire Floor of Brownstone                 |31         |false    |false   |
|Entire Apt: Spacious Studio/Loft by central park|48         |true     |false   |
|Large Cozy 1 BR Apartment In Midtown East       |41         |false    |false   |
|BlissArtsSpace!                                 |15         |false    |false   |
|Large Furnished Room Near B'way                 |32         |false    |false   |
|Cozy Clean Guest Room - Family Apt              |34         |false    |false   |
|Cute & Cozy Low

In [8]:
# Engenharia de Atributos sobre Datas: Extração de componentes temporais
df_datas = df_texto.withColumn(
    "review_year", 
    F.coalesce(F.year(F.col("last_review")), F.lit(0))
).withColumn(
    "review_month", 
    F.coalesce(F.month(F.col("last_review")), F.lit(0))
).withColumn(
    "review_day_of_week", 
    F.coalesce(F.dayofweek(F.col("last_review")), F.lit(0))
).withColumn(
    "is_weekend_review", 
    F.coalesce(F.dayofweek(F.col("last_review")).isin([1, 7]), F.lit(False))
).withColumn(
    "reviews_per_month_clean", 
    F.coalesce(F.col("reviews_per_month"), F.lit(0.0))
).withColumn(
    "has_any_review", 
    F.when(F.col("last_review").isNull(), False).otherwise(True)
)


In [9]:
# Selecionando a coluna original e as novas colunas temporais
df_datas.select("last_review", "review_year", "review_month", "review_day_of_week", "is_weekend_review", "reviews_per_month", "last_review").show()

# Nota de Validação: Colunas originais ('last_review' e 'reviews_per_month') preservam os nulos para auditoria, enquanto as novas colunas limpas contêm os dados tratados.

+-----------+-----------+------------+------------------+-----------------+-----------------+-----------+
|last_review|review_year|review_month|review_day_of_week|is_weekend_review|reviews_per_month|last_review|
+-----------+-----------+------------+------------------+-----------------+-----------------+-----------+
| 2018-10-19|       2018|          10|                 6|            false|             0.21| 2018-10-19|
| 2019-05-21|       2019|           5|                 3|            false|             0.38| 2019-05-21|
|       NULL|          0|           0|                 0|            false|             NULL|       NULL|
| 2019-07-05|       2019|           7|                 6|            false|             4.64| 2019-07-05|
| 2018-11-19|       2018|          11|                 2|            false|              0.1| 2018-11-19|
| 2019-06-22|       2019|           6|                 7|             true|             0.59| 2019-06-22|
| 2017-10-05|       2017|          10|        

In [10]:
# Segmentação de Negócio e Escalonamento: Agrupamento do tempo mínimo de estadia em categorias de mercado e normalização da escala de preços
df_numerico = df_datas.withColumn(
    "log_price", 
    F.log(F.col("price") + 1)
).withColumn(
    "estadia_tipo",
    F.when(F.col("minimum_nights") <= 2, "Curta (1-2 noites)")
     .when((F.col("minimum_nights") > 2) & (F.col("minimum_nights") <= 7), "Media (3-7 noites)")
     .otherwise("Longa (7+ noites)"))

# Justificativa técnica para o log_price:
# 1. Reduz a assimetria (skewness) causada por imóveis muito caros (outliers), aproximando os preços de uma distribuição normal.
# 2. O uso de log(price + 1) evita erros matemáticos caso existam preços iguais a zero, pois log(0 + 1) = 0.

In [11]:
# Inspeção e validação do efeito da curva logarítmica sobre os preços e da distribuição dos binnings de estadia
df_numerico.select("price", "log_price", "minimum_nights", "estadia_tipo").show(truncate=False)

+-----+------------------+--------------+------------------+
|price|log_price         |minimum_nights|estadia_tipo      |
+-----+------------------+--------------+------------------+
|149  |5.0106352940962555|1             |Curta (1-2 noites)|
|225  |5.420534999272286 |1             |Curta (1-2 noites)|
|150  |5.017279836814924 |3             |Media (3-7 noites)|
|89   |4.499809670330265 |1             |Curta (1-2 noites)|
|80   |4.394449154672439 |10            |Longa (7+ noites) |
|200  |5.303304908059076 |3             |Media (3-7 noites)|
|60   |4.110873864173311 |45            |Longa (7+ noites) |
|79   |4.382026634673881 |2             |Curta (1-2 noites)|
|79   |4.382026634673881 |2             |Curta (1-2 noites)|
|150  |5.017279836814924 |1             |Curta (1-2 noites)|
|135  |4.912654885736052 |5             |Media (3-7 noites)|
|85   |4.454347296253507 |2             |Curta (1-2 noites)|
|89   |4.499809670330265 |4             |Media (3-7 noites)|
|85   |4.454347296253507

In [12]:
# Análise via Window Functions
from pyspark.sql.window import Window

# 1. Definimos a "janela" (grupo) baseada na região do imóvel
window_regiao = Window.partitionBy("neighbourhood_group")

# 2. Criamos as novas colunas olhando para essa janela
df_final = df_numerico.withColumn(
    "avg_price_neighbourhood",
    F.avg(F.col("price")).over(window_regiao)
).withColumn(
    # Razão entre o preço do imóvel e a média da região dele
    "price_ratio_neighbourhood",
    F.col("price") / F.col("avg_price_neighbourhood")
)

"""
EXPLICAÇÃO TÉCNICA: POR QUE USAR WINDOW FUNCTIONS AQUI?

1. Agregação sem Perda de Granularidade: Diferente de um 'groupBy' tradicional, 
   que colapsa e reduz o número de linhas do DataFrame, a Window Function calcula 
   a média regional (avg) mas mantém o nível de detalhe de cada imóvel (linha por linha).

2. Cálculo Inter-Linhas Eficiente: Permite comparar o valor individual de um registro 
   diretamente com a métrica do seu grupo correspondente ('neighbourhood_group'), 
   viabilizando operações imediatas como a razão de preço ('price_ratio_neighbourhood') 
   sem a necessidade de realizar um Join custoso entre dois DataFrames.
"""

"\nEXPLICAÇÃO TÉCNICA: POR QUE USAR WINDOW FUNCTIONS AQUI?\n\n1. Agregação sem Perda de Granularidade: Diferente de um 'groupBy' tradicional, \n   que colapsa e reduz o número de linhas do DataFrame, a Window Function calcula \n   a média regional (avg) mas mantém o nível de detalhe de cada imóvel (linha por linha).\n\n2. Cálculo Inter-Linhas Eficiente: Permite comparar o valor individual de um registro \n   diretamente com a métrica do seu grupo correspondente ('neighbourhood_group'), \n   viabilizando operações imediatas como a razão de preço ('price_ratio_neighbourhood') \n   sem a necessidade de realizar um Join custoso entre dois DataFrames.\n"

In [13]:
# Inspeção analítica das métricas regionais para avaliar o desvio e a proporção de preço de cada imóvel em relação à sua média local
df_final.select(
    "neighbourhood_group", 
    "price", 
    "avg_price_neighbourhood", 
    "price_ratio_neighbourhood"
).show()

+-------------------+-----+-----------------------+-------------------------+
|neighbourhood_group|price|avg_price_neighbourhood|price_ratio_neighbourhood|
+-------------------+-----+-----------------------+-------------------------+
|              Bronx|   40|       87.4967919340055|      0.45715961826543333|
|              Bronx|   45|       87.4967919340055|       0.5143045705486124|
|              Bronx|   90|       87.4967919340055|        1.028609141097225|
|              Bronx|  105|       87.4967919340055|       1.2000439979467625|
|              Bronx|   90|       87.4967919340055|        1.028609141097225|
|              Bronx|   77|       87.4967919340055|       0.8800322651609591|
|              Bronx|   37|       87.4967919340055|      0.42287264689552584|
|              Bronx|  125|       87.4967919340055|       1.4286238070794792|
|              Bronx|   50|       87.4967919340055|       0.5714495228317916|
|              Bronx|   50|       87.4967919340055|       0.5714

In [14]:
# Persistência de Dados: Escrita dos dados transformados no formato colunar Parquet com estratégia de sobrescrita (overwrite) para otimização de consultas futuras
df_final.write.mode("overwrite").parquet("/opt/spark/resources/NYC/base/airbnb_feature_store.parquet")

print("Base salva!")

2026-06-22T01:37:19,716 [Thread-3] WARN  org.apache.spark.sql.catalyst.util.SparkStringUtils [] - Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Base salva!


In [15]:
# Finalização do Pipeline: Encerramento da SparkSession para liberação dos recursos alocados no cluster (memória e CPU)
spark.stop()

In [16]:
%reload_ext watermark
%watermark -a "André Luiz Colombo"

Author: André Luiz Colombo



In [17]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.8.10
IPython version      : 8.12.3

Compiler    : GCC 9.4.0
OS          : Linux
Release     : 6.12.76-linuxkit
Machine     : x86_64
Processor   : x86_64
CPU cores   : 12
Architecture: 64bit



# Fim